In [5]:
from threading import Thread

## Single Thread

In [6]:
def my_function():
    print("printing from thread")
    
if __name__ == "__main__":
    thread = Thread(target=my_function)
    thread.start()
    thread.join()


printing from thread


## Multiple Threads

In [7]:
from threading import Thread
def my_function():
    print("printing from thread")
if __name__ == "__main__":
    threads = [Thread(target=my_function) for _ in range(10)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

printing from thread
printing from thread
printing from thread
printing from thread
printing from thread
printing from thread
printing from thread
printing from thread
printing from thread
printing from thread


## Multi-thread same context

In [10]:
from threading import Thread
thread_visits = 0
def visit_counter():
    global thread_visits
    for i in range(100_000):
        value = thread_visits
        thread_visits = value + 1
if __name__ == "__main__":
    thread_count = 100
    threads = [
        Thread(target=visit_counter)
        for _ in range(thread_count)
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    print(f"{thread_count=}, {thread_visits=}")

thread_count=100, thread_visits=10000000


## Multi-thread locking context

In [12]:
from threading import Lock
thread_visits = 0
thread_visits_lock = Lock()
def visit_counter():
    global thread_visits
    for i in range(100_000):
        with thread_visits_lock:  # executes the operation within a lock
            thread_visits += 1
if __name__ == "__main__":
    thread_count = 100
    threads = [
        Thread(target=visit_counter)
        for _ in range(thread_count)
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    print(f"{thread_count=}, {thread_visits=}")

thread_count=100, thread_visits=10000000


## Multi-thread example

In [24]:
import requests
response = requests.get("https://api.vatcomply.com/rates?base=USD")
response.json()

{'date': '2022-08-31',
 'base': 'USD',
 'rates': {'EUR': 1.0,
  'USD': 1.0,
  'JPY': 138.72,
  'BGN': 1.9558,
  'CZK': 24.55,
  'DKK': 7.4371,
  'GBP': 0.86035,
  'HUF': 402.8,
  'PLN': 4.7283,
  'RON': 4.8595,
  'SEK': 10.6788,
  'CHF': 0.9796,
  'ISK': 141.7,
  'NOK': 9.9388,
  'HRK': 7.5148,
  'TRY': 18.1849,
  'AUD': 1.4591,
  'BRL': 5.1482,
  'CAD': 1.3111,
  'CNY': 6.8947,
  'HKD': 7.8488,
  'IDR': 14849.93,
  'ILS': 3.3399,
  'INR': 79.5465,
  'KRW': 1342.79,
  'MXN': 20.2044,
  'MYR': 4.4755,
  'NZD': 1.6322,
  'PHP': 56.153,
  'SGD': 1.3969,
  'THB': 36.45,
  'ZAR': 17.0667}}

In [25]:
import time
import requests
SYMBOLS = ('USD', 'EUR', 'PLN', 'NOK', 'CZK')
BASES = ('USD', 'EUR', 'PLN', 'NOK', 'CZK')
def fetch_rates(base):
    response = requests.get(
        f"https://api.vatcomply.com/rates?base={base}"
    )
    response.raise_for_status()
    rates = response.json()["rates"]
    
    # note: same currency exchanges to itself 1:1
    rates[base] = 1.
    rates_line = ", ".join(
        [f"{rates[symbol]:7.03} {symbol}" for symbol in SYMBOLS]
    )
    print(f"1 {base} = {rates_line}")



In [27]:
# non-threaded solution

def main():
    for base in BASES:
        fetch_rates(base)
if __name__ == "__main__":
    started = time.time()
    main()
    elapsed = time.time() - started
    print()
    print("time elapsed: {:.2f}s".format(elapsed))

1 USD =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 EUR =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 PLN =   0.211 USD,   0.211 EUR,     1.0 PLN,     2.1 NOK,    5.19 CZK
1 NOK =   0.101 USD,   0.101 EUR,   0.476 PLN,     1.0 NOK,    2.47 CZK
1 CZK =  0.0407 USD,  0.0407 EUR,   0.193 PLN,   0.405 NOK,     1.0 CZK

time elapsed: 3.52s


In [28]:
#tyhreaded solution

from threading import Thread
def main():
    threads = []
    for base in BASES:
        thread = Thread(target=fetch_rates, args=[base])
        thread.start()
        threads.append(thread)
    while threads:
        threads.pop().join()

if __name__ == "__main__":
    started = time.time()
    main()
    elapsed = time.time() - started
    print()
    print("time elapsed: {:.2f}s".format(elapsed))


1 NOK =   0.101 USD,   0.101 EUR,   0.476 PLN,     1.0 NOK,    2.47 CZK
1 USD =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 EUR =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 PLN =   0.211 USD,   0.211 EUR,     1.0 PLN,     2.1 NOK,    5.19 CZK
1 CZK =  0.0407 USD,  0.0407 EUR,   0.193 PLN,   0.405 NOK,     1.0 CZK

time elapsed: 1.08s


## Pool thread

In [29]:
from queue import Queue
from threading import Thread
THREAD_POOL_SIZE = 4


from queue import Empty

def worker(work_queue):
    while not work_queue.empty():
        try:
            item = work_queue.get_nowait()
        except Empty:
            break
        else:
            fetch_rates(item)
            work_queue.task_done()

def main():
    work_queue = Queue()
    for base in BASES:
        work_queue.put(base)
    threads = [
        Thread(target=worker, args=(work_queue,))
        for _ in range(THREAD_POOL_SIZE)
    ]
    for thread in threads:
        thread.start()
    work_queue.join()
    while threads:
        threads.pop().join()
        
if __name__ == "__main__":
    started = time.time()
    main()
    elapsed = time.time() - started
    print()
    print("time elapsed: {:.2f}s".format(elapsed))

1 NOK =   0.101 USD,   0.101 EUR,   0.476 PLN,     1.0 NOK,    2.47 CZK
1 USD =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 EUR =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 PLN =   0.211 USD,   0.211 EUR,     1.0 PLN,     2.1 NOK,    5.19 CZK
1 CZK =  0.0407 USD,  0.0407 EUR,   0.193 PLN,   0.405 NOK,     1.0 CZK

time elapsed: 1.45s


## Pooled no print

In [30]:
import time
from queue import Queue, Empty
from threading import Thread
import requests
SYMBOLS = ('USD', 'EUR', 'PLN', 'NOK', 'CZK')
BASES = ('USD', 'EUR', 'PLN', 'NOK', 'CZK')
THREAD_POOL_SIZE = 4
def fetch_rates(base):
    response = requests.get(
        f"https://api.vatcomply.com/rates?base={base}"
    )
    response.raise_for_status()
    rates = response.json()["rates"]
    # note: same currency exchanges to itself 1:1
    rates[base] = 1.
    return base, rates
def present_result(base, rates):
    rates_line = ", ".join([
        f"{rates[symbol]:7.03} {symbol}" 
        for symbol in SYMBOLS
    ])
    print(f"1 {base} = {rates_line}")
def worker(work_queue, results_queue):
    while not work_queue.empty():
        try:
            item = work_queue.get_nowait()
        except Empty:
            break
        else:
            results_queue.put(fetch_rates(item))
            work_queue.task_done()
def main():
    work_queue = Queue()
    results_queue = Queue()
    for base in BASES:
        work_queue.put(base)
    threads = [
        Thread(
            target=worker, 
            args=(work_queue, results_queue)
        ) for _ in range(THREAD_POOL_SIZE)
    ]
    for thread in threads:
        thread.start()
    work_queue.join()
    while threads:
        threads.pop().join()
    while not results_queue.empty():
        present_result(*results_queue.get())
if __name__ == "__main__":
    started = time.time()
    main()
    elapsed = time.time() - started
    print()
    print("time elapsed: {:.2f}s".format(elapsed)) 

1 PLN =   0.211 USD,   0.211 EUR,     1.0 PLN,     2.1 NOK,    5.19 CZK
1 NOK =   0.101 USD,   0.101 EUR,   0.476 PLN,     1.0 NOK,    2.47 CZK
1 EUR =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 USD =     1.0 USD,     1.0 EUR,    4.73 PLN,    9.94 NOK,    24.6 CZK
1 CZK =  0.0407 USD,  0.0407 EUR,   0.193 PLN,   0.405 NOK,     1.0 CZK

time elapsed: 1.44s
